<center><ins><h1>Sinking Velocity</h1></ins></center>

<h5>Definition:</h5>
<ul>
    <li>Sinking velocities of marine particles are generally determined by two major properties—size and excess density over seawater. Most organic particles within the oceans are relatively small (< 100 μm) and their individual contribution to the downward flux of organic matter is therefore minor (Clegg and Whitfield 1990).</li>
    <li>Unit is not defined yet (in m/h or m/s or else?).</li>
    <li>Files are imported over .csv in individual sample files. Create Dataframe with Sample Information + Value mean Sinking Velocity (scalar_factor = 3.674)</li>
    <li>Applied Filter: filter(sincing_velocity in px/s != negative) => filter(gof > 0.6) => if gof.count < 10 {mean_sv = 0} else {mean_sv = data}</li>
</ul>

<center>
<img src="../images/sinking-velocity_device.png" alt="Device" width="536" height="300">
<img src="../images/sinking-velocity_diagram.png" alt="Diagram" width="376" height="300">
<img src="../images/sinking-velocity_example-graph.png" alt="Example-Graph" width="289" height="300">
</center>

###### **Author:** Cedric Hering-Peter
###### **Address:** AG Schulz / Botantical Institut / Am Botanischen Garten 5 / 24118 Kiel

# Packages

In [1]:
# Imports
import os, sys
sys.path.append(os.path.abspath("/Users/cedric/Documents/Career/PhD-Student/Experiments/Data-Science"))
from scripts import data_helper, plot_helper
import pandas as pd

# Variables

In [2]:
GOF_THRESHOLD = 0.6
COUNT_THRESHOLD = 25
CALIBRATION_FACTOR = 3.674
META_DATA = pd.read_csv("../data/meta_data.csv")

# Show numeric output in decimal format e.g., 2.15
pd.options.display.float_format = '{:,.2f}'.format

# Functions

In [3]:
def filter_particles(unfil_df):
    fil_df = unfil_df.loc[unfil_df["sincing_velocity in px/s"] >= 0.0] # Filter raising particles
    fil_df = fil_df.loc[fil_df["gof"] > GOF_THRESHOLD] # Sinking quality filter
    fil_df = fil_df.loc[fil_df["first x position"] < fil_df["last x position"]] # Filter particles that were still raising
    if len(fil_df) < COUNT_THRESHOLD: 
        return fil_df[0:0] # Return empty data frame if to less particles are in the data 
    return fil_df

def create_entry(file):
    # Insert sample information in a new empty data frame
    entry = pd.DataFrame(columns = META_DATA["SAMPLE_INFORMATION"].dropna().tolist())
    entry.loc[0] = os.path.basename(file).split(".csv")[0].split("_")
    
    # Filter invalid data values from file
    unfil_df = pd.read_csv(file, sep = ";", keep_default_na = False)
    fil_df = filter_particles(unfil_df)

    # Custom data calculation from current sample
    particle_count = len(fil_df) # Total amount of valid particles
    # Particle loss due to filtering, if not 0 particles in current sample
    if not fil_df.empty: 
        particle_loss = 100 - ((len(fil_df) / len(unfil_df)) * 100)
    else: 
        particle_loss = 0
    mean_sv = ((fil_df["sincing_velocity in px/s"].mean() / 10**6) * 3600) * CALIBRATION_FACTOR # Conversion into Meter per Hour
    median_sv = ((fil_df["sincing_velocity in px/s"].median() / 10**6) * 3600) * CALIBRATION_FACTOR # Conversion into Meter per Hour
    
    # Add data value columns to entry
    entry["PARTICLE_COUNT_N"] = particle_count 
    entry["PARTICLE_LOSS_%"] = particle_loss
    entry["MEAN_SVELO_M/H"] = mean_sv
    entry["MEDIAN_SVELO_M/H"] = median_sv
    return entry

def custom_svelo_import(files):
    df = pd.DataFrame()
    for file in files:
        # Append new entry row to raw data frame
        df = pd.concat([df, create_entry(file)], ignore_index = True)
    return df

# Raw Import

In [4]:
# Define data path and get all associated files
data_dir = "/Users/cedric/Documents/Career/PhD-Student/Experiments/Instrument-Data/Flow-Cam_Sinki-Vinci"
files = data_helper.create_filelist(data_dir, skip = " ")

# Import raw data frame
raw_df = custom_svelo_import(files)
# Convert data types
raw_df = data_helper.convert_types(raw_df)
# Append start controls to physiology samples    
raw_df = data_helper.copy_start_control(raw_df, META_DATA)
# Sort data
raw_df = raw_df.sort_values(by = ["EXPERIMENT_NAME", "TIME", "SAMPLE_NAME", "BIO_REP", "TEC_REP"], ignore_index = True)
raw_df.head(3)

,ID,EXPERIMENT_NAME,SAMPLE_NAME,TIME,BIO_REP,TEC_REP,PARTICLE_COUNT_N,PARTICLE_LOSS_%,MEAN_SVELO_M/H,MEDIAN_SVELO_M/H
0,CH230710,AT,0.1,0,1,1,1050,31.73,2.51,2.52
1,CH230802,AT,0.1,0,2,1,360,35.14,3.15,3.10
2,CH230907,AT,0.1,0,3,1,1195,30.24,1.99,1.79


# Custom Cleaning

In [5]:
clean_df = pd.DataFrame(raw_df)

# Drop duplicate measurements 
clean_df = clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "5-Second-Run"].index)
# Add possible additional sinking units
#clean_df["MEAN_SV_M/S"] = clean_df["MEAN_SV_µM/S"] / 3600 # Meter per Second
#clean_df["MEDIAN_SV_M/S"] = clean_df["MEDIAN_SV_µM/S"] / 3600 # Meter per Second
#clean_df["MEAN_SV_M/D"] = (clean_df["MEAN_SV_µM/S"] / 10**6) * (3600 * 24) # Meter per Day
#clean_df["MEDIAN_SV_M/D"] = (clean_df["MEDIAN_SV_µM/S"] / 10**6) * (3600 * 24) # Meter per Day

# Lower species count from 7 to 5
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "C.vulgaris"].index, inplace = True)
clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "P.duplex"].index, inplace = True)

clean_df.drop(clean_df[clean_df["SAMPLE_NAME"] == "+EPS"].index, inplace = True)

print(f"{raw_df.shape[0] - clean_df.shape[0]} rows and {raw_df.shape[1] - clean_df.shape[1]} cols were cleaned.")

22 rows and 0 cols were cleaned.


# File Export

In [6]:
# Export to raw data to csv-file
export_df = pd.DataFrame(clean_df)
file_name = "../exports/Raw-Data/Sinking-Velocities_Raw-Data.csv"
export_df.to_csv(file_name, encoding = "utf-8", index = False)

# Plot Visualization 

In [7]:
# Filter time before plotting
plot_df = pd.DataFrame(data_helper.filter(clean_df, time = [12]))

# Iterate through all experiment, visualize and save plots
for exp in META_DATA["EXP_ABBR"].unique():
    sub_df = plot_df[plot_df["EXPERIMENT_NAME"] == exp]
    sub_meta = META_DATA[META_DATA["EXP_ABBR"] == exp][:1].reset_index()
    
    plot_helper.visualize_boxplot(sub_df, sub_meta, # current data subset
                                  show_x_label = True,
                                  y_data = "MEAN_SVELO_M/H", # Toggle mean / median
                                  y_label = "Sinking Velocity [m\u0020" + r"${h^{-1}}$]", # Without unit serves as file name
                                  y_ticks = [0, 6, 1],
                                  y_labelpad = 49.5)

Boxplot "Species Composition" created and saved.
Boxplot "Salinity Treatment" created and saved.
Boxplot "pH Treatment" created and saved.
Boxplot "Antibiotics Treatment" created and saved.
Boxplot "Light Treatment" created and saved.
Boxplot "Temperature Treatment" created and saved.
Boxplot "Culture Composition" created and saved.
Boxplot "Reflocculation" created and saved.
Boxplot "Bioflocculation" created and saved.


# ARCHIVE